# Execução Pipelines

https://www.researchgate.net/publication/318919520_Anomaly_Detection_in_Streams_with_Extreme_Value_Theory

## Otimização

In [ ]:
%load_ext autoreload
%autoreload 2

import optuna
import pandas as pd
import time
from src.Anomaly.Pipeline import AnomalyExperimentRunner
from src.Anomaly.Models import get_anomaly_models
from src.Data.Processor import DataStreamProcessor
optuna.logging.set_verbosity(optuna.logging.WARNING)

categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '250', '500', '1000', '2000']
datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

features = [
    'Min Packet Length',
    'act_data_pkt_fwd',
    'Subflow Fwd Bytes',
    'Fwd Packet Length Min',
    'Total Length of Fwd Packets',
    'Packet Length Mean',
    'Average Packet Size',
    'Total Fwd Packets',
    'Avg Fwd Segment Size',
    'Fwd Packets/s',
    'Fwd Packet Length Mean',
    'Max Packet Length',
    'Init_Win_bytes_forward',
    'Fwd Packet Length Max',
    'Inbound',
    'Subflow Fwd Packets',
    'Subflow Bwd Packets',
    'Flow IAT Mean',
    'URG Flag Count',
    'Fwd IAT Mean',
    'Flow IAT Max',
    'Total Backward Packets',
    'Init_Win_bytes_backward',
    'Flow IAT Std',
    'ACK Flag Count',
]

processor = DataStreamProcessor(logging=False, selected_features=features)

algoritmos_para_testar = ['AIF']

def objective(trial, dataset_path, alg_name):
    dspot_q = trial.suggest_float('q', 1e-6, 1e-1, log=True)
    dspot_depth = trial.suggest_int('depth', 100, 400)
    dspot_t = trial.suggest_float('t_quantile', 0.93, 0.99)

    try:
        df = pd.read_csv(dataset_path)
    except FileNotFoundError:
        raise optuna.exceptions.TrialPruned()

    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol'],
        imputation_method='mediana'
    )

    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=[alg_name]
    )

    runner = AnomalyExperimentRunner(target_names=targets)
    titulo_dinamico = f"Trial {trial.number} | {alg_name} | q: {dspot_q:.5f} | d: {dspot_depth} | t: {dspot_t:.3f}"
    
    resultados = runner._run_anomaly_DSPOT(
        stream=stream,
        algorithms=algoritmos,
        window_size=2048,
        warmup_instances=1000,
        title=titulo_dinamico,
        target_class=None,
        dspot_q=dspot_q,
        dspot_depth=dspot_depth,
        dspot_t_quantile=dspot_t 
    )
    

    f1_val = resultados.get('f1_score', 0.0)
    prec_val = resultados.get('precision', 0.0)
    rec_val = resultados.get('recall', 0.0)

    # SALVANDO MÉTRICAS EXTRAS NO OPTUNA
    trial.set_user_attr('precision', prec_val)
    trial.set_user_attr('recall', rec_val)

    print(f"[{alg_name}] Trial {trial.number:03d}")
    print(f"   -> Params  | q: {dspot_q:.6f} | depth: {dspot_depth} | t_quantile: {dspot_t:.4f}")
    print(f"   -> Métricas| F1: {f1_val:.2f}% | Prec: {prec_val:.2f}% | Rec: {rec_val:.2f}%\n")

    return f1_val

melhores_parametros_geral = {}

for alg in algoritmos_para_testar:
    print(f"\n{'='*70}")
    print(f" INICIANDO OTIMIZAÇÃO: ALGORITMO {alg} ")
    print(f"{'='*70}")
    melhores_parametros_geral[alg] = {}
    
    for dataset_path in datasets:
        print(f"\n--- Processando Dataset: {dataset_path.split('/')[-1]} ---\n")
        
        study_name = f"{alg}_{dataset_path.replace('/', '_')}"
        study = optuna.create_study(direction='maximize', study_name=study_name)
        
        study.optimize(lambda trial: objective(trial, dataset_path, alg), n_trials=50) 
        
        best_trial = study.best_trial
        
        melhores_parametros_geral[alg][dataset_path] = {
            'F1-Score': best_trial.value,
            'Precision': best_trial.user_attrs.get('precision', 0.0),
            'Recall': best_trial.user_attrs.get('recall', 0.0),
            'Tempo Execução (s)': best_trial.user_attrs.get('exec_time', 0.0),
            'q': best_trial.params['q'],
            'depth': best_trial.params['depth'],
            't_quantile': best_trial.params['t_quantile'],
            'Trial_Vencedor': best_trial.number
        }

print("\n\n" + "#"*70)
print(" RELATÓRIO FINAL DA OTIMIZAÇÃO DSPOT ".center(70, "#"))
print("#"*70)

for alg, datasets_results in melhores_parametros_geral.items():
    print(f"\n>> Algoritmo: {alg}")
    print("-" * 40)
    for ds, metrics in datasets_results.items():
        print(f"Dataset: {ds.split('/')[-1]} (Melhor Trial: {metrics['Trial_Vencedor']:03d})")
        print(f"  Métricas  -> F1-Score: {metrics['F1-Score']:.2f}% | Precision: {metrics['Precision']:.2f}% | Recall: {metrics['Recall']:.2f}%")
        print(f"  Parâmetros-> q: {metrics['q']:.6f} | depth: {metrics['depth']} | t_quantile: {metrics['t_quantile']:.4f}")
        print(f"  Tempo     -> {metrics['Tempo Execução (s)']:.2f} segundos")

## Treino DSPOT

In [ ]:
%load_ext autoreload
%autoreload 2

from src.Anomaly.Pipeline import AnomalyExperimentRunner
from src.Anomaly.Models import get_anomaly_models
from src.Data.Processor import DataStreamProcessor
import pandas as pd

categorias = ['Consistência']
tamanhos = ['25', '250', '500', '1000', '2000']
datasets = [f'data/50k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

features = [
    'Min Packet Length',
    'act_data_pkt_fwd',
    'Subflow Fwd Bytes',
    'Fwd Packet Length Min',
    'Total Length of Fwd Packets',
    'Packet Length Mean',
    'Average Packet Size',
    'Total Fwd Packets',
    'Avg Fwd Segment Size',
    'Fwd Packets/s',
    'Fwd Packet Length Mean',
    'Max Packet Length',
    'Init_Win_bytes_forward',
    'Fwd Packet Length Max',
    'Inbound',
    'Subflow Fwd Packets',
    'Subflow Bwd Packets',
    'Flow IAT Mean',
    'URG Flag Count',
    'Fwd IAT Mean',
    'Flow IAT Max',
    'Total Backward Packets',
    'Init_Win_bytes_backward',
    'Flow IAT Std',
    'ACK Flag Count',
]

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=features)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AIF']
    )
    
    runner = AnomalyExperimentRunner(target_names=targets)
    
    runner._run_anomaly_DSPOT(
        stream,
        algorithms=algoritmos,
        window_size=2048,
        warmup_instances=1000,
        title=nome_experimento,
        target_class=None,
        dspot_q=0.00001,
        dspot_depth=151,
        dspot_t_quantile=0.96
    )

# Comparação

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.Anomaly.Pipeline import AnomalyExperimentRunner
from src.Anomaly.Models import get_anomaly_models
from src.Data.Processor import DataStreamProcessor

categoria = 'Recorrência'
tamanhos = ['25', '250', '500', '1000', '2000']
processor = DataStreamProcessor(logging=False)

def criar_cenarios(categoria):
    scenarios = {}
    targets_globais = None
    
    for tam in tamanhos:
        dataset_path = f'data/15k/{categoria}/{categoria}_{tam}.csv'
        
        try:
            df = pd.read_csv(dataset_path)
        except FileNotFoundError:
            continue
            
        stream, targets, _ = processor.create_stream(
            df=df, 
            target_label_col='Label', 
            binary_label=False, 
            normalize_method="MinMaxScaler", 
            return_stream=True,
            extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol'], 
            imputation_method='mediana'
        )
        targets_globais = targets
        
        algoritmos = get_anomaly_models(
            stream.get_schema(), 
            selected_models=['AIF']
        )
        nome_algoritmo = list(algoritmos.keys())[0]
        
        scenarios[tam] = {'stream': stream, 'learner': algoritmos[nome_algoritmo]}
        
    return scenarios, targets_globais

# Avaliação do Threshold Fixo
scenarios_fixo, targets_globais = criar_cenarios(categoria)
runner = AnomalyExperimentRunner(target_names=targets_globais)

resultados_fixo = runner._run_poisoning_evolution(
    scenarios_dict=scenarios_fixo, 
    threshold_type='fixed', 
    fixed_threshold=0.6
)

# Avaliação do DSPOT
scenarios_dspot, _ = criar_cenarios(categoria)

resultados_dspot = runner._run_poisoning_evolution(
    scenarios_dict=scenarios_dspot, 
    threshold_type='dspot', 
    warmup_instances=1000, 
    dspot_q=0.004, 
    dspot_depth=748
)

# Plotagem Lado a Lado
runner.plots.plot_poisoning_evolution(
    results_fixed=resultados_fixo,
    results_dspot=resultados_dspot,
    title=f'{categoria} - Adaptative Isolation Forest',
    fixed_threshold=0.6,
    window_size=5
)